In [5]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
import numpy as np
import pandas as pd
import tensorflow as tf
from MockBroker import MockBroker
# import matplotlib.pyplot as plt
import plotly.graph_objects as go



models = {}
norm_layers = {}

nifty_symbols = [
    "ADANIPORTS", "ASIANPAINT", "AXISBANK", "BAJAJFINSV",
    "BAJFINANCE", "BPCL", "BRITANNIA", "CIPLA", "COALINDIA",
    "DIVISLAB", 
    "DRREDDY", "EICHERMOT", "GRASIM", "HCLTECH",
    "HDFCBANK", "HDFCLIFE", "HEROMOTOCO", "HINDALCO", 
    "HINDUNILVR",
    "ICICIBANK", "ICICIGI", "IOC", "INDUSINDBK", "INFY",
    "ITC", "JSWSTEEL", "KOTAKBANK", "LTTS", "LT",
    "MARICO",
      "MARUTI", "NESTLEIND"
]

# Load models
for symbol in nifty_symbols:
    models[symbol] = load_model(
        f'./models/{symbol}_model.keras', 
        # custom_objects={'Orthogonal': Orthogonal}
    )

data_test = {}
data_pred = {}
price = {}

threshold = 0.001
days= 500
cash =10000
loss_thres = 1000

for symbol in nifty_symbols:
    # Load the test data
    price[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, 0].values
    data_test[symbol] = pd.read_csv(f'./data/{symbol}.csv').iloc[-days:, [0,1,2,3]].values

    # Normalize the test data
    norm_l = Normalization(axis=-1)
    norm_l.adapt(data_test[symbol])  # Ensure normalization layer adapts to the test data
    data_test_n = norm_l(data_test[symbol])

    # Reshape the data for LSTM input
    data_test_n = np.reshape(data_test_n, (data_test_n.shape[0], 1, data_test_n.shape[1]))

    # Predict using the model
    data_pred[symbol] = models[symbol].predict(data_test_n)

    






16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
16/16 ━━━━━━━━━━━━━━━━━━

In [6]:


# Dictionary to store test data and predictions


# Load the test data
# Normalization layer (reusing the one from training)
# Initialize the broker
broker = MockBroker(cash, price)



for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres(i,loss_thres)


    for symbol, prediction in top_3_symbols[:1]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)



--- Day 0 ---
[('LT', 0.00302129), ('AXISBANK', 0.0025776299), ('BAJAJFINSV', 0.0018268265)]

Networth: 10000
Bought 4.0 of LT at 2093.5 each
{'LT': 4}
Balance: 1626.0
--- Day 1 ---
[('LT', 0.0021818425), ('DRREDDY', 0.002080835), ('ASIANPAINT', 0.0016871641)]
Sold 4 of LT at 2124.0 each
{'LT': 0}
Balance: 10122.0

Networth: 10122.0
Bought 4.0 of LT at 2124.0 each
{'LT': 4}
Balance: 1626.0
--- Day 2 ---
[('LT', 0.0018158614), ('DRREDDY', 0.0017499344), ('BAJAJFINSV', 0.0016677971)]
Sold 4 of LT at 2167.7 each
{'LT': 0}
Balance: 10296.8

Networth: 10296.8
Bought 4.0 of LT at 2167.7 each
{'LT': 4}
Balance: 1626.0
--- Day 3 ---
[('LT', 0.0016893514), ('ASIANPAINT', 0.0016338466), ('BAJAJFINSV', 0.0014741379)]
Sold 4 of LT at 2154.05 each
{'LT': 0}
Balance: 10242.2

Networth: 10242.2
Bought 4.0 of LT at 2154.05 each
{'LT': 4}
Balance: 1626.0
--- Day 4 ---
[('LT', 0.0016048726), ('BAJAJFINSV', 0.0015408727), ('KOTAKBANK', 0.0015365975)]
Sold 4 of LT at 2163.7 each
{'LT': 0}
Balance: 10280.8

Equity: 0.0
 Holdings: {'LT': 0, 'BAJAJFINSV': 0, 'LTTS': 0, 'KOTAKBANK': 0, 'INDUSINDBK': 0, 'DRREDDY': 0, 'ICICIGI': 0, 'HDFCLIFE': 0, 'DIVISLAB': 3, 'AXISBANK': 0, 'CIPLA': 0, 'MARICO': 0, 'HINDALCO': 0, 'ASIANPAINT': 0, 'EICHERMOT': 0, 'HINDUNILVR': 0, 'HCLTECH': 0, 'JSWSTEEL': 0}
 Balance: 447.032180000002
 Total: 18244.832180000005
 Highest Net Worth: 21972.53218000001
 Stocks Traded: {'ICICIGI', 'LTTS', 'LT', 'HINDUNILVR', 'HCLTECH', 'AXISBANK', 'MARICO', 'DRREDDY', 'EICHERMOT', 'INDUSINDBK', 'JSWSTEEL', 'HINDALCO', 'ASIANPAINT', 'BAJAJFINSV', 'KOTAKBANK', 'CIPLA', 'HDFCLIFE', 'DIVISLAB'}
 Profit off Stocks: {'LT': 3068.0, 'BAJAJFINSV': 1063.0, 'LTTS': 1558.0, 'KOTAKBANK': 875.0, 'INDUSINDBK': 378.0, 'DRREDDY': 748.0, 'ICICIGI': 2062.0, 'HDFCLIFE': -1156.0, 'DIVISLAB': -17797.800000000003, 'AXISBANK': 135.0, 'CIPLA': 1498.0, 'MARICO': 143.79999999999995, 'HINDALCO': 119.0, 'ASIANPAINT': -1015.7000000000007, 'EICHERMOT': -1812.0, 'HINDUNILVR': 47.0, 'HCLTECH': 550.0, 'JSWSTEEL': 

In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Normalization
from tensorflow.keras.initializers import Orthogonal
from MockBroker import MockBroker


# Initialize the broker



# Dictionary to store test data and predictions


broker = MockBroker(cash, price)


# Load the test data
# Normalization layer (reusing the one from training)




for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres(i, loss_thres)



    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        qty = np.floor(broker.Balance / (3*current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)



x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('LT', 0.00302129), ('AXISBANK', 0.0025776299), ('BAJAJFINSV', 0.0018268265)]

Networth: 10000
Bought 1.0 of LT at 2093.5 each
{'LT': 1}
Balance: 7906.5
Bought 2.0 of AXISBANK at 904.65002 each
{'LT': 1, 'AXISBANK': 2}
Balance: 6097.19996
Bought 1.0 of BAJAJFINSV at 1635.25 each
{'LT': 1, 'AXISBANK': 2, 'BAJAJFINSV': 1}
Balance: 4461.94996
--- Day 1 ---
[('LT', 0.0021818425), ('DRREDDY', 0.002080835), ('ASIANPAINT', 0.0016871641)]
Sold 1 of LT at 2124.0 each
{'LT': 0, 'AXISBANK': 2, 'BAJAJFINSV': 1}
Balance: 6585.94996
Sold 2 of AXISBANK at 914.65002 each
{'LT': 0, 'AXISBANK': 0, 'BAJAJFINSV': 1}
Balance: 8415.25

Networth: 10021.95
Bought 1.0 of LT at 2124.0 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1}
Balance: 6291.25
Bought 2.0 of DRREDDY at 872.3 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1, 'DRREDDY': 2}
Balance: 4546.65
Bought 0.0 of ASIANPAINT at 3226.5 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1, 'DRREDDY': 2, 'ASIANPAINT': 0}
Balance: 4546.65
--- Day 2 ---


Equity: 2440.18
 Holdings: {'LT': 0, 'AXISBANK': 0, 'BAJAJFINSV': 1, 'DRREDDY': 0, 'ASIANPAINT': 0, 'KOTAKBANK': 0, 'ICICIGI': 0, 'LTTS': 0, 'NESTLEIND': 0, 'MARUTI': 0, 'INDUSINDBK': 0, 'HINDALCO': 0, 'HDFCLIFE': 0, 'MARICO': 0, 'DIVISLAB': 0, 'CIPLA': 0, 'JSWSTEEL': 0, 'EICHERMOT': 0, 'ITC': 0, 'HDFCBANK': 0, 'INFY': 0, 'HINDUNILVR': 0, 'COALINDIA': 0, 'HCLTECH': 0, 'IOC': 17, 'BRITANNIA': 0}
 Balance: 6923.095710000012
 Total: 11026.675710000012
 Highest Net Worth: 12209.015710000007
 Stocks Traded: {'ITC', 'LT', 'INFY', 'JSWSTEEL', 'HINDALCO', 'ASIANPAINT', 'KOTAKBANK', 'MARUTI', 'AXISBANK', 'EICHERMOT', 'HDFCLIFE', 'HDFCBANK', 'LTTS', 'HCLTECH', 'MARICO', 'INDUSINDBK', 'CIPLA', 'DIVISLAB', 'BRITANNIA', 'ICICIGI', 'HINDUNILVR', 'COALINDIA', 'DRREDDY', 'IOC', 'BAJAJFINSV', 'NESTLEIND'}
 Profit off Stocks: {'LT': 202.0, 'AXISBANK': 88.0, 'BAJAJFINSV': -2805.4, 'DRREDDY': 270.0, 'ASIANPAINT': 45.0, 'KOTAKBANK': 236.0, 'ICICIGI': 527.0, 'LTTS': 0.0, 'NESTLEIND': 3.0, 'MARUTI': 0.0, 'IN

In [8]:

# Dictionary to store test data and predictions


broker = MockBroker(cash, price)



# Load the test data
# Normalization layer (reusing the one from training)


for i in range(days):
    print(f"--- Day {i} ---")
    predictions = []

    for symbol in nifty_symbols:
        predictions.append((symbol, data_pred[symbol][i][0]))

    # Sort the predictions by the prediction value
    predictions.sort(key=lambda x: x[1], reverse=True)

    # Get the top 3 symbols with the highest prediction values
    top_3_symbols = predictions[:3]
    print(top_3_symbols)
    broker.sell_off_if_thres(i, loss_thres)

    
    sumprices = 0

    for symbol, prediction in top_3_symbols[:3]:
            sumprices += price[symbol][i]
    
    # print(sumprices)

    for symbol, prediction in top_3_symbols[:3]:
        current_price = price[symbol][i]
        balance_alloted = broker.Balance*current_price/sumprices
        qty = np.floor(balance_alloted / (current_price))
        if prediction > threshold:
            broker.buy(symbol, current_price, qty)


x = np.linspace(0, days, days)  # 100 points between 0 and 10
y = broker.networth

fig = go.Figure()

# Add a line plot to the figure
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Net Worth',  # Legend label
    line=dict(color='blue', width=2)  # Line styling
))

# Add title and labels to the plot
fig.update_layout(
    title='Net Worth Over Time',
    xaxis_title='Days',
    yaxis_title='Net Worth (USD)',
    template='plotly_dark'  # Optional: Add a dark theme for the plot
)

# Show the plot
fig.show()

print(broker)


--- Day 0 ---
[('LT', 0.00302129), ('AXISBANK', 0.0025776299), ('BAJAJFINSV', 0.0018268265)]

Networth: 10000
Bought 2.0 of LT at 2093.5 each
{'LT': 2}
Balance: 5813.0
Bought 1.0 of AXISBANK at 904.65002 each
{'LT': 2, 'AXISBANK': 1}
Balance: 4908.34998
Bought 1.0 of BAJAJFINSV at 1635.25 each
{'LT': 2, 'AXISBANK': 1, 'BAJAJFINSV': 1}
Balance: 3273.09998
--- Day 1 ---
[('LT', 0.0021818425), ('DRREDDY', 0.002080835), ('ASIANPAINT', 0.0016871641)]
Sold 2 of LT at 2124.0 each
{'LT': 0, 'AXISBANK': 1, 'BAJAJFINSV': 1}
Balance: 7521.09998
Sold 1 of AXISBANK at 914.65002 each
{'LT': 0, 'AXISBANK': 0, 'BAJAJFINSV': 1}
Balance: 8435.75

Networth: 10042.45
Bought 1.0 of LT at 2124.0 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1}
Balance: 6311.75
Bought 1.0 of DRREDDY at 872.3 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1, 'DRREDDY': 1}
Balance: 5439.45
Bought 0.0 of ASIANPAINT at 3226.5 each
{'LT': 1, 'AXISBANK': 0, 'BAJAJFINSV': 1, 'DRREDDY': 1, 'ASIANPAINT': 0}
Balance: 5439.45
--- Day 2 ---


Equity: 0.0
 Holdings: {'LT': 0, 'AXISBANK': 0, 'BAJAJFINSV': 0, 'DRREDDY': 0, 'ASIANPAINT': 0, 'KOTAKBANK': 0, 'ICICIGI': 0, 'LTTS': 0, 'NESTLEIND': 0, 'MARUTI': 0, 'INDUSINDBK': 0, 'HINDALCO': 0, 'HDFCLIFE': 0, 'MARICO': 0, 'DIVISLAB': 0, 'CIPLA': 0, 'JSWSTEEL': 0, 'EICHERMOT': 0, 'ITC': 0, 'HDFCBANK': 0, 'INFY': 0, 'HINDUNILVR': 0, 'COALINDIA': 0, 'HCLTECH': 0, 'IOC': 0, 'BRITANNIA': 0}
 Balance: 10878.629159999997
 Total: 10878.629159999997
 Highest Net Worth: 11320.419160000001
 Stocks Traded: {'ITC', 'LT', 'INFY', 'JSWSTEEL', 'HINDALCO', 'ASIANPAINT', 'KOTAKBANK', 'MARUTI', 'AXISBANK', 'EICHERMOT', 'HDFCLIFE', 'HDFCBANK', 'LTTS', 'HCLTECH', 'MARICO', 'INDUSINDBK', 'CIPLA', 'DIVISLAB', 'BRITANNIA', 'ICICIGI', 'HINDUNILVR', 'COALINDIA', 'DRREDDY', 'IOC', 'BAJAJFINSV', 'NESTLEIND'}
 Profit off Stocks: {'LT': 501.0, 'AXISBANK': 30.0, 'BAJAJFINSV': 201.0, 'DRREDDY': 81.0, 'ASIANPAINT': -1074.0, 'KOTAKBANK': 33.0, 'ICICIGI': 317.0, 'LTTS': 141.0, 'NESTLEIND': 0.0, 'MARUTI': 0.0, 'INDUS